## 1. Extract and Validate Daily FX Rates

Extract daily ECB exchange rates from the intermediate Gold Taxi Fact table.

Validate the number of records and ensure each calendar date has a unique exchange rate.

In [ ]:
df_gold_fx = spark.table("dbo.gold_fact_taxi_daily")

df_gold_fx.select(
    "trip_date",
    "usd_per_eur_filled"
).distinct().orderBy("trip_date").show(10)



In [ ]:
df_fx_daily = df_gold_fx.select(
    "trip_date",
    "usd_per_eur_filled"
).distinct()

print("FX rows:", df_fx_daily.count())

print(
    "Unique dates:",
    df_fx_daily.select("trip_date").distinct().count()
)

## 2. Prepare FX Dimension

Rename the date and exchange rate columns to create a consistent schema for the Gold FX table.

Verify the final schema and exchange rate values.

In [ ]:
df_fx_daily = df_fx_daily.withColumnRenamed(
    "trip_date",
    "rate_date"
).withColumnRenamed(
    "usd_per_eur_filled",
    "usd_per_eur"
)

In [ ]:
df_fx_daily.printSchema()
df_fx_daily.orderBy("rate_date").show(5)

## 3. Save and Verify Gold FX Table

Save the prepared daily exchange rates as a Delta table in the Gold layer and verify the persisted records.

In [ ]:
df_fx_daily.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbo.gold_fx_daily")

In [ ]:
df_saved_fx = spark.table("dbo.gold_fx_daily")

df_saved_fx.orderBy("rate_date").show(5)
print("Saved rows:", df_saved_fx.count())

In [ ]:
# df_air = spark.table("dbo.silver_air_quality")

# df_air.printSchema()

# df_air.show(5, truncate=False)

# print("Total rows:", df_air.count())

In [ ]:
# from pyspark.sql import functions as F

# df_air.groupBy("pollutant").agg(
#     F.count("*").alias("total_rows"),
#     F.countDistinct("sensor_id").alias("sensors"),
#     F.min("measurement_date").alias("start_date"),
#     F.max("measurement_date").alias("end_date")
# ).show(truncate=False)

In [ ]:
# df_daily_counts = df_air.groupBy("measurement_date").agg(
#     F.count("*").alias("hourly_records")
# )

# df_daily_counts.filter(
#     F.col("hourly_records") != 24
# ).orderBy("measurement_date").show()

In [ ]:
# df_duplicates = (
#     df_air
#     .groupBy("sensor_id", "measurement_time_local")
#     .count()
#     .filter(F.col("count") > 1)
# )

# df_duplicates.show()

# print("Duplicate timestamps:", df_duplicates.count())